# AE3: preparación y análisis descriptivo de salarios

## Necesidad

Este ejercicio académico plantea una necesidad hipotética del equipo de talento humano: comparar los salarios en dólares registrados para cargos de datos y tecnología, según la experiencia, la modalidad de trabajo y el tamaño de la empresa. No representa una solicitud de una empresa real ni un problema comprobado.

## Preguntas de análisis

1. ¿Cómo se comparan los salarios en USD (salary_in_usd) por nivel de experiencia, modalidad remota y tamaño de empresa?
2. ¿Cuántos registros hay por año de referencia (work_year), nivel de experiencia y tipo de empleo?
3. ¿Qué cargos tienen más registros y cuál es su salario mediano en USD?
4. ¿Cómo cambia el salario mediano en USD entre los años cubiertos?

## Variables y resultados esperados

| Pregunta | Variables | Resultado |
|---|---|---|
| Principal | salary_in_usd, experience_level, remote_ratio, company_size | Conteo y resumen salarial por combinación de grupos. |
| Apoyo 1 | work_year, experience_level, employment_type | Conteo por año, experiencia y tipo de empleo. |
| Apoyo 2 | job_title, salary_in_usd | Conteo y salario mediano por cargo. |
| Apoyo 3 | work_year, salary_in_usd | Mediana por año y variación frente al año anterior. |

El dataset tiene work_year, que indica un año de referencia, no una fecha de pago exacta. El análisis es descriptivo y utiliza el CSV de AE2, que se conserva sin modificar.

## 1. Cargar los datos y conservar el original

El archivo demo-salaries-original.csv es la copia sin modificar del dataset de AE2. En esta sección lo cargaremos y revisaremos sus dimensiones, tipos de datos, valores faltantes y registros repetidos antes de aplicar transformaciones.

In [1]:
from pathlib import Path

import pandas as pd
from IPython.display import display

# Buscar la raíz del proyecto aunque el notebook se abra desde otra carpeta.
ROOT = Path.cwd().resolve()

while ROOT != ROOT.parent and not (
    ROOT / "src" / "Entrega_3" / "datos" / "demo-salaries-original.csv"
).is_file():
    ROOT = ROOT.parent

RUTA_ORIGINAL = (
    ROOT / "src" / "Entrega_3" / "datos" / "demo-salaries-original.csv"
)

if not RUTA_ORIGINAL.is_file():
    raise FileNotFoundError(f"No se encontró el archivo original: {RUTA_ORIGINAL}")

# Cargar y conservar los datos originales sin modificarlos.
df_original = pd.read_csv(RUTA_ORIGINAL)
df = df_original.copy(deep=True)

print("Ruta del archivo:", RUTA_ORIGINAL)
print("Dimensiones originales:", df_original.shape)
print("\nTipos de columnas:")
display(df_original.dtypes.to_frame("tipo"))
print("\nValores nulos por columna:")
display(df_original.isna().sum().to_frame("valores_nulos"))
print("\nFilas repetidas después de la primera aparición:",
      int(df_original.duplicated().sum()))
print("\nPrimeras cinco filas:")
display(df_original.head())

Ruta del archivo: /workspaces/Proyecto_analitica_3/src/Entrega_3/datos/demo-salaries-original.csv
Dimensiones originales: (3755, 11)

Tipos de columnas:


,tipo
work_year,int64
experience_level,object
employment_type,object
job_title,object
salary,int64
salary_currency,object
salary_in_usd,int64
employee_residence,object
remote_ratio,int64
company_location,object



Valores nulos por columna:


,valores_nulos
work_year,0
experience_level,0
employment_type,0
job_title,0
salary,0
salary_currency,0
salary_in_usd,0
employee_residence,0
remote_ratio,0
company_location,0



Filas repetidas después de la primera aparición: 1171

Primeras cinco filas:


,work_year,experience_level,employment_type,job_title,salary,salary_currency,salary_in_usd,employee_residence,remote_ratio,company_location,company_size
0,2023,SE,FT,Principal Data Scientist,80000,EUR,85847,ES,100,ES,L
1,2023,MI,CT,ML Engineer,30000,USD,30000,US,100,US,S
2,2023,MI,CT,ML Engineer,25500,USD,25500,US,100,US,S
3,2023,SE,FT,Data Scientist,175000,USD,175000,CA,100,CA,M
4,2023,SE,FT,Data Scientist,120000,USD,120000,CA,100,CA,M


## 2. Transformar y validar los datos

Se conservarán todos los registros y no se reemplazarán valores faltantes. Se comprobarán las columnas necesarias y los valores de salario en USD. Después se cambiarán los códigos de experiencia, empleo y tamaño de empresa por etiquetas más claras, y se creará remote_work_mode a partir de remote_ratio. Al final compararemos las dimensiones, los nulos y los duplicados con el archivo original.

In [2]:
# Comprobar que existan las columnas necesarias.
columnas_requeridas = {
    "work_year",
    "experience_level",
    "employment_type",
    "salary",
    "salary_in_usd",
    "remote_ratio",
    "company_size",
}

faltantes = columnas_requeridas - set(df.columns)
if faltantes:
    raise ValueError(f"Faltan columnas requeridas: {sorted(faltantes)}")

# Asegurar que estas variables numéricas sean enteros.
for columna in ["work_year", "salary", "salary_in_usd", "remote_ratio"]:
    df.loc[:, columna] = pd.to_numeric(df[columna], errors="raise").astype("Int64")

# Reemplazar códigos por etiquetas legibles.
categorias = {
    "experience_level": {
        "EN": "Inicial",
        "MI": "Intermedio",
        "SE": "Senior",
        "EX": "Ejecutivo",
    },
    "employment_type": {
        "FT": "Tiempo completo",
        "PT": "Tiempo parcial",
        "CT": "Contrato",
        "FL": "Freelance",
    },
    "company_size": {
        "S": "Pequeña",
        "M": "Mediana",
        "L": "Grande",
    },
}

for columna, equivalencias in categorias.items():
    valores = df[columna].astype("string").str.strip()
    desconocidos = set(valores.dropna()) - set(equivalencias)

    if desconocidos:
        raise ValueError(
            f"Códigos no contemplados en {columna}: {sorted(desconocidos)}"
        )

    df.loc[:, columna] = valores.map(equivalencias)

# Derivar la modalidad laboral del porcentaje remoto.
modalidades = {
    0: "Presencial",
    50: "Híbrido",
    100: "Remoto",
}

ratios_desconocidos = set(df["remote_ratio"].dropna().astype(int)) - set(modalidades)
if ratios_desconocidos:
    raise ValueError(f"Valores inesperados en remote_ratio: {sorted(ratios_desconocidos)}")

df.loc[:, "remote_work_mode"] = df["remote_ratio"].map(modalidades)

# Revisar que el salario en dólares no sea negativo.
if (df["salary_in_usd"].dropna() < 0).any():
    raise ValueError("Se encontraron salarios negativos en salary_in_usd.")

# Comparar el original y el DataFrame transformado.
comparacion = pd.DataFrame(
    [
        {
            "version": "Original",
            "filas": len(df_original),
            "columnas": len(df_original.columns),
            "nulos": int(df_original.isna().sum().sum()),
            "filas_repetidas": int(df_original.duplicated().sum()),
        },
        {
            "version": "Transformado",
            "filas": len(df),
            "columnas": len(df.columns),
            "nulos": int(df.isna().sum().sum()),
            "filas_repetidas": int(df.duplicated().sum()),
        },
    ]
)

display(comparacion)
display(df.head())

,version,filas,columnas,nulos,filas_repetidas
0,Original,3755,11,0,1171
1,Transformado,3755,12,0,1171


,work_year,experience_level,employment_type,job_title,salary,salary_currency,salary_in_usd,employee_residence,remote_ratio,company_location,company_size,remote_work_mode
0,2023,Senior,Tiempo completo,Principal Data Scientist,80000,EUR,85847,ES,100,ES,Grande,Remoto
1,2023,Intermedio,Contrato,ML Engineer,30000,USD,30000,US,100,US,Pequeña,Remoto
2,2023,Intermedio,Contrato,ML Engineer,25500,USD,25500,US,100,US,Pequeña,Remoto
3,2023,Senior,Tiempo completo,Data Scientist,175000,USD,175000,CA,100,CA,Mediana,Remoto
4,2023,Senior,Tiempo completo,Data Scientist,120000,USD,120000,CA,100,CA,Mediana,Remoto


## 3. Pregunta principal: comparación de salarios por grupos

¿Cómo se comparan los salarios en USD según el nivel de experiencia, la modalidad de trabajo remoto y el tamaño de la empresa?

Agruparemos los registros por esas tres variables. Para cada combinación calcularemos el número de registros, la mediana y el promedio del salario, además de los cuartiles 25 y 75. El conteo ayuda a interpretar los salarios de cada grupo. Solo se mostrarán las combinaciones presentes en los datos.

In [3]:
# Resumir los salarios por cada combinación observada.
tabla_principal = (
    df.groupby(
        ["experience_level", "remote_work_mode", "company_size"],
        dropna=False,
        observed=True,
    )
    .agg(
        cantidad_registros=("salary_in_usd", "size"),
        salario_usd_mediana=("salary_in_usd", "median"),
        salario_usd_promedio=("salary_in_usd", "mean"),
        salario_usd_q1=("salary_in_usd", lambda valores: valores.quantile(0.25)),
        salario_usd_q3=("salary_in_usd", lambda valores: valores.quantile(0.75)),
    )
    .reset_index()
    .sort_values(
        ["experience_level", "remote_work_mode", "company_size"]
    )
    .reset_index(drop=True)
)

# Confirmar que los grupos incluyen todos los registros.
total_contabilizado = int(tabla_principal["cantidad_registros"].sum())
assert total_contabilizado == len(df), (
    f"La tabla cuenta {total_contabilizado} registros, pero el DataFrame tiene {len(df)}."
)

print(f"Combinaciones observadas: {len(tabla_principal)}")
print(f"Registros contabilizados: {total_contabilizado}")
display(tabla_principal)

Combinaciones observadas: 34
Registros contabilizados: 3755


,experience_level,remote_work_mode,company_size,cantidad_registros,salario_usd_mediana,salario_usd_promedio,salario_usd_q1,salario_usd_q3
0,Ejecutivo,Híbrido,Grande,5,125976.0,140831.000000,79833.00,192037.00
1,Ejecutivo,Híbrido,Mediana,1,130026.0,130026.000000,130026.00,130026.00
2,Ejecutivo,Presencial,Grande,4,123923.0,145474.750000,100513.25,168884.50
3,Ejecutivo,Presencial,Mediana,52,181080.0,185278.942308,145000.00,212500.00
4,Ejecutivo,Remoto,Grande,4,194333.5,215916.750000,152750.25,257500.00
5,Ejecutivo,Remoto,Mediana,42,218500.0,217307.404762,161218.75,255500.00
6,Ejecutivo,Remoto,Pequeña,6,190000.0,196827.166667,131416.50,200000.00
7,Inicial,Híbrido,Grande,30,50000.0,79801.333333,37463.50,95632.00
8,Inicial,Híbrido,Mediana,22,30760.0,42780.227273,22106.00,56826.50
9,Inicial,Híbrido,Pequeña,13,55000.0,63476.923077,28609.00,100000.00


### Interpretación y límites

La tabla permite comparar la mediana, el promedio y los cuartiles del salario en USD para las combinaciones de experiencia, modalidad de trabajo y tamaño de empresa que aparecen en los datos. El número de registros ayuda a ver cuántas observaciones sustentan cada comparación.

Estas diferencias describen únicamente este dataset. No demuestran que la experiencia, la modalidad o el tamaño de la empresa causen diferencias salariales ni representan necesariamente el mercado. Conviene tener cuidado al interpretar grupos con pocos registros.

In [4]:
# Guardar la tabla principal para reutilizarla después.
CARPETA_RESULTADOS = ROOT / "src" / "Entrega_3" / "resultados"
CARPETA_RESULTADOS.mkdir(parents=True, exist_ok=True)

ruta_tabla_principal = (
    CARPETA_RESULTADOS / "pregunta_principal_salarios_por_grupo.csv"
)
tabla_principal.to_csv(
    ruta_tabla_principal,
    index=False,
    encoding="utf-8-sig",
)

# Resumir algunos resultados de la tabla.
grupo_mayor_mediana = tabla_principal.loc[
    tabla_principal["salario_usd_mediana"].idxmax()
]
grupo_mas_registros = tabla_principal.loc[
    tabla_principal["cantidad_registros"].idxmax()
]

print(
    "Mediana más alta observada: "
    f"{grupo_mayor_mediana['experience_level']}, "
    f"{grupo_mayor_mediana['remote_work_mode']}, "
    f"empresa {grupo_mayor_mediana['company_size']}: "
    f"USD {grupo_mayor_mediana['salario_usd_mediana']:,.0f} "
    f"(n={int(grupo_mayor_mediana['cantidad_registros'])})."
)
print(
    "Grupo con más registros: "
    f"{grupo_mas_registros['experience_level']}, "
    f"{grupo_mas_registros['remote_work_mode']}, "
    f"empresa {grupo_mas_registros['company_size']}: "
    f"{int(grupo_mas_registros['cantidad_registros'])} registros."
)
print(f"\nTabla exportada a: {ruta_tabla_principal}")

Mediana más alta observada: Ejecutivo, Remoto, empresa Mediana: USD 218,500 (n=42).
Grupo con más registros: Senior, Presencial, empresa Mediana: 1269 registros.

Tabla exportada a: /workspaces/Proyecto_analitica_3/src/Entrega_3/resultados/pregunta_principal_salarios_por_grupo.csv


## 4. Pregunta de apoyo 1: registros por año, experiencia y tipo de empleo

¿Cuántos registros hay por año de referencia, nivel de experiencia y tipo de empleo?

Usaremos work_year como año de referencia, ya que el dataset no incluye una fecha exacta de pago. Contaremos los registros para cada combinación observada.

In [5]:
# Contar registros por año, experiencia y tipo de empleo.
tabla_apoyo_1 = (
    df.groupby(
        ["work_year", "experience_level", "employment_type"],
        dropna=False,
        observed=True,
    )
    .size()
    .reset_index(name="cantidad_registros")
    .sort_values(
        ["work_year", "experience_level", "employment_type"]
    )
    .reset_index(drop=True)
)

# Comprobar que la tabla cuenta todas las filas.
total = int(tabla_apoyo_1["cantidad_registros"].sum())
assert total == len(df), f"La tabla cuenta {total}, pero el DataFrame tiene {len(df)} filas."

display(tabla_apoyo_1)
print(f"Combinaciones observadas: {len(tabla_apoyo_1)}")
print(f"Registros contabilizados: {total}")

# Exportar la tabla a CSV.
ruta_apoyo_1 = CARPETA_RESULTADOS / (
    "pregunta_apoyo_1_registros_por_anio_experiencia_empleo.csv"
)
tabla_apoyo_1.to_csv(
    ruta_apoyo_1,
    index=False,
    encoding="utf-8-sig",
)
print(f"Tabla exportada a: {ruta_apoyo_1}")

,work_year,experience_level,employment_type,cantidad_registros
0,2020,Ejecutivo,Tiempo completo,3
1,2020,Inicial,Contrato,1
2,2020,Inicial,Tiempo completo,20
3,2020,Inicial,Tiempo parcial,2
4,2020,Intermedio,Tiempo completo,32
5,2020,Senior,Freelance,1
6,2020,Senior,Tiempo completo,17
7,2021,Ejecutivo,Contrato,1
8,2021,Ejecutivo,Tiempo completo,9
9,2021,Inicial,Tiempo completo,50


Combinaciones observadas: 35
Registros contabilizados: 3755
Tabla exportada a: /workspaces/Proyecto_analitica_3/src/Entrega_3/resultados/pregunta_apoyo_1_registros_por_anio_experiencia_empleo.csv


### Interpretación y límites

La tabla muestra cuántos registros hay para cada combinación observada de año de referencia, nivel de experiencia y tipo de empleo. En el conjunto completo predominan los registros de tiempo completo: 3.718 de 3.755. Estos conteos describen el dataset; no representan necesariamente la distribución del mercado laboral. Algunos grupos pueden tener pocos registros, por lo que conviene interpretarlos con cautela.

work_year es el año de referencia disponible, no una fecha exacta de pago.

## 5. Pregunta de apoyo 2: cargos con más registros y salario mediano

¿Qué cargos tienen más registros y cuál es su salario mediano en USD?

Agruparemos los registros por cargo y calcularemos su cantidad y la mediana del salario en dólares. La tabla completa se guardará en CSV; aquí mostraremos los diez cargos más frecuentes.

In [6]:
# Contar registros y calcular la mediana salarial por cargo.
tabla_apoyo_2 = (
    df.groupby("job_title", dropna=False, observed=True)
    .agg(
        cantidad_registros=("salary_in_usd", "size"),
        salario_usd_mediana=("salary_in_usd", "median"),
    )
    .reset_index()
    .sort_values(
        ["cantidad_registros", "job_title"],
        ascending=[False, True],
    )
    .reset_index(drop=True)
)

# Comprobar que se contabilizaron todas las filas.
total = int(tabla_apoyo_2["cantidad_registros"].sum())
assert total == len(df), (
    f"La tabla cuenta {total}, pero el DataFrame tiene {len(df)} filas."
)

# Mostrar los diez cargos más frecuentes.
display(tabla_apoyo_2.head(10))
print(f"Cargos en la tabla completa: {len(tabla_apoyo_2)}")
print(f"Registros contabilizados: {total}")

# Exportar la tabla completa para reutilizarla.
ruta_apoyo_2 = CARPETA_RESULTADOS / (
    "pregunta_apoyo_2_cargos_por_registros_y_mediana.csv"
)
tabla_apoyo_2.to_csv(
    ruta_apoyo_2,
    index=False,
    encoding="utf-8-sig",
)
print(f"Tabla completa exportada a: {ruta_apoyo_2}")

,job_title,cantidad_registros,salario_usd_mediana
0,Data Engineer,1040,135000.0
1,Data Scientist,840,141525.0
2,Data Analyst,612,106020.0
3,Machine Learning Engineer,289,150000.0
4,Analytics Engineer,103,143860.0
5,Data Architect,101,160000.0
6,Research Scientist,82,150000.0
7,Applied Scientist,58,191737.5
8,Data Science Manager,58,183780.0
9,Research Engineer,37,155000.0


Cargos en la tabla completa: 93
Registros contabilizados: 3755
Tabla completa exportada a: /workspaces/Proyecto_analitica_3/src/Entrega_3/resultados/pregunta_apoyo_2_cargos_por_registros_y_mediana.csv


### Interpretación y límites

Data Engineer es el cargo con más registros (1.040), con un salario mediano de USD 135.000. Le sigue Data Scientist, con 840 registros y una mediana de USD 141.525. La tabla CSV contiene los resultados de todos los cargos.

Estas cifras describen únicamente los registros del dataset. La cantidad de observaciones varía entre cargos, por lo que las medianas de grupos pequeños deben interpretarse con cautela. No son recomendaciones salariales ni representan necesariamente el mercado.

## 6. Pregunta de apoyo 3: salario mediano por año

¿Cómo cambia el salario mediano en USD de un año a otro?

Agruparemos por work_year y calcularemos la cantidad de registros y la mediana salarial de cada año. También calcularemos la diferencia en USD y el cambio porcentual respecto al año anterior. Para el primer año, esos cambios quedarán vacíos porque no tenemos un año previo en la tabla.

In [7]:
# Calcular registros y salario mediano por año de referencia.
tabla_apoyo_3 = (
    df.groupby("work_year", dropna=False, observed=True)
    .agg(
        cantidad_registros=("salary_in_usd", "size"),
        salario_usd_mediano=("salary_in_usd", "median"),
    )
    .reset_index()
    .sort_values("work_year")
    .reset_index(drop=True)
)

# Calcular diferencias respecto al año previo.
tabla_apoyo_3.loc[:, "variacion_mediana_usd_vs_anio_anterior"] = (
    tabla_apoyo_3["salario_usd_mediano"].diff()
)
tabla_apoyo_3.loc[:, "variacion_mediana_porcentual_vs_anio_anterior"] = (
    tabla_apoyo_3["salario_usd_mediano"].pct_change() * 100
)

# Verificar que se contabilizaron todas las filas.
total = int(tabla_apoyo_3["cantidad_registros"].sum())
assert total == len(df), (
    f"La tabla cuenta {total}, pero el DataFrame tiene {len(df)} filas."
)

formato_usd = lambda valor: f"{valor:,.0f}".replace(",", ".")

display(
    tabla_apoyo_3.style.format({
        "salario_usd_mediano": formato_usd,
        "variacion_mediana_usd_vs_anio_anterior": (
            lambda valor: "—" if pd.isna(valor) else formato_usd(valor)
        ),
        "variacion_mediana_porcentual_vs_anio_anterior": (
            lambda valor: "—" if pd.isna(valor) else f"{valor:.1f} %"
        ),
    })
)
print(f"Registros contabilizados: {total}")

# Exportar la tabla a CSV.
ruta_apoyo_3 = CARPETA_RESULTADOS / (
    "pregunta_apoyo_3_salario_mediano_por_anio.csv"
)
tabla_apoyo_3.to_csv(
    ruta_apoyo_3,
    index=False,
    encoding="utf-8-sig",
)
print(f"Tabla exportada a: {ruta_apoyo_3}")

,work_year,cantidad_registros,salario_usd_mediano,variacion_mediana_usd_vs_anio_anterior,variacion_mediana_porcentual_vs_anio_anterior
0,2020,76,73.065,—,—
1,2021,230,80.000,6.935,9.5 %
2,2022,1664,131.300,51.300,64.1 %
3,2023,1785,143.860,12.560,9.6 %


Registros contabilizados: 3755
Tabla exportada a: /workspaces/Proyecto_analitica_3/src/Entrega_3/resultados/pregunta_apoyo_3_salario_mediano_por_anio.csv


### Interpretación y límites

La mediana salarial registrada fue de USD 73.065 en 2020, USD 80.000 en 2021, USD 131.300 en 2022 y USD 143.860 en 2023. El mayor aumento interanual observado fue entre 2021 y 2022: USD 51.300 (64,1 %).

Estos cambios describen las medianas de los registros disponibles; no demuestran una evolución del mercado ni sus causas. El número de registros también varía entre años, por lo que la composición de la muestra puede influir en la comparación.

## 7. Exportar los resultados y describir el DataFrame

Guardaremos el DataFrame transformado, las cuatro tablas y un resumen descriptivo en archivos. El resumen se genera desde df en cada ejecución e incluye dimensiones, tipos de columnas, nulos y estadísticas descriptivas. Al final comprobaremos que los archivos existan y que los conteos de las tablas coincidan con el total de registros.

In [8]:
# Crear la carpeta de resultados y guardar el DataFrame preparado.
CARPETA_DATOS = ROOT / "src" / "Entrega_3" / "datos"
CARPETA_RESULTADOS = ROOT / "src" / "Entrega_3" / "resultados"

CARPETA_DATOS.mkdir(parents=True, exist_ok=True)
CARPETA_RESULTADOS.mkdir(parents=True, exist_ok=True)

ruta_preparado = CARPETA_DATOS / "demo-salaries-preparado.csv"
df.to_csv(ruta_preparado, index=False, encoding="utf-8-sig")

# Relacionar cada tabla con el nombre de su archivo.
tablas_resultados = {
    "pregunta_principal_salarios_por_grupo": tabla_principal,
    "pregunta_apoyo_1_registros_por_anio_experiencia_empleo": tabla_apoyo_1,
    "pregunta_apoyo_2_cargos_por_registros_y_mediana": tabla_apoyo_2,
    "pregunta_apoyo_3_salario_mediano_por_anio": tabla_apoyo_3,
}

# Comprobar conteos y exportar cada tabla.
for nombre, tabla in tablas_resultados.items():
    if int(tabla["cantidad_registros"].sum()) != len(df):
        raise ValueError(f"Los conteos de {nombre} no suman {len(df)}.")

    ruta = CARPETA_RESULTADOS / f"{nombre}.csv"
    tabla.to_csv(ruta, index=False, encoding="utf-8-sig")
    print(f"Tabla guardada: {ruta}")

# Generar de nuevo el reporte descriptivo desde el DataFrame final.
lineas = [
    "RESUMEN DEL DATAFRAME PREPARADO",
    "=" * 40,
    f"Filas: {len(df)}",
    f"Columnas: {len(df.columns)}",
    f"Filas repetidas: {int(df.duplicated().sum())}",
    "",
    "Columnas y tipos:",
]

for columna, tipo in df.dtypes.items():
    lineas.append(f"- {columna}: {tipo}")

lineas.extend(["", "Valores nulos por columna:"])
for columna, cantidad in df.isna().sum().items():
    lineas.append(f"- {columna}: {cantidad}")

lineas.extend([
    "",
    "Estadísticas descriptivas:",
    df.describe(include="all").transpose().to_string(),
])

ruta_resumen = CARPETA_RESULTADOS / "resumen_dataframe.txt"
ruta_resumen.write_text("\n".join(lineas) + "\n", encoding="utf-8")

# Comprobar que los archivos generados coinciden en dimensiones y conteos.
df_exportado = pd.read_csv(ruta_preparado)
assert df_exportado.shape == df.shape, "El CSV preparado no coincide con df."

for nombre, tabla in tablas_resultados.items():
    tabla_exportada = pd.read_csv(
        CARPETA_RESULTADOS / f"{nombre}.csv"
    )
    assert len(tabla_exportada) == len(tabla)
    assert int(tabla_exportada["cantidad_registros"].sum()) == len(df)

print(f"\nDataset preparado: {ruta_preparado} ({df_exportado.shape[0]} filas)")
print(f"Resumen actualizado: {ruta_resumen}")
print("Verificación completada: los conteos de las cuatro tablas suman", len(df))

Tabla guardada: /workspaces/Proyecto_analitica_3/src/Entrega_3/resultados/pregunta_principal_salarios_por_grupo.csv
Tabla guardada: /workspaces/Proyecto_analitica_3/src/Entrega_3/resultados/pregunta_apoyo_1_registros_por_anio_experiencia_empleo.csv
Tabla guardada: /workspaces/Proyecto_analitica_3/src/Entrega_3/resultados/pregunta_apoyo_2_cargos_por_registros_y_mediana.csv
Tabla guardada: /workspaces/Proyecto_analitica_3/src/Entrega_3/resultados/pregunta_apoyo_3_salario_mediano_por_anio.csv

Dataset preparado: /workspaces/Proyecto_analitica_3/src/Entrega_3/datos/demo-salaries-preparado.csv (3755 filas)
Resumen actualizado: /workspaces/Proyecto_analitica_3/src/Entrega_3/resultados/resumen_dataframe.txt
Verificación completada: los conteos de las cuatro tablas suman 3755


## 8. Diccionario actualizado de variables

La tabla siguiente describe las variables del DataFrame preparado, su tipo de dato y su significado. remote_work_mode es una variable derivada de remote_ratio. La periodicidad de los salarios no está especificada en el dataset.

In [9]:
# Describir las variables originales y la variable derivada.
significados = {
    "work_year": ("Año de referencia del registro", "Año"),
    "experience_level": ("Nivel de experiencia", "Categoría"),
    "employment_type": ("Tipo de empleo", "Categoría"),
    "job_title": ("Cargo registrado", "Texto"),
    "salary": ("Salario en su moneda original", "Moneda indicada en salary_currency"),
    "salary_currency": ("Código de moneda del salario original", "Código de moneda"),
    "salary_in_usd": ("Salario convertido a dólares estadounidenses", "USD; periodicidad no especificada"),
    "employee_residence": ("País de residencia del empleado", "Código de país"),
    "remote_ratio": ("Porcentaje de trabajo remoto", "%: 0, 50 o 100"),
    "company_location": ("País de ubicación de la empresa", "Código de país"),
    "company_size": ("Tamaño de la empresa", "Categoría"),
    "remote_work_mode": ("Modalidad derivada de remote_ratio", "0 presencial; 50 híbrido; 100 remoto"),
}

diccionario = pd.DataFrame([
    {
        "variable": columna,
        "tipo_de_dato": str(df[columna].dtype),
        "significado": significados[columna][0],
        "unidad_o_regla": significados[columna][1],
    }
    for columna in df.columns
])

display(diccionario)

,variable,tipo_de_dato,significado,unidad_o_regla
0,work_year,int64,Año de referencia del registro,Año
1,experience_level,object,Nivel de experiencia,Categoría
2,employment_type,object,Tipo de empleo,Categoría
3,job_title,object,Cargo registrado,Texto
4,salary,int64,Salario en su moneda original,Moneda indicada en salary_currency
5,salary_currency,object,Código de moneda del salario original,Código de moneda
6,salary_in_usd,int64,Salario convertido a dólares estadounidenses,USD; periodicidad no especificada
7,employee_residence,object,País de residencia del empleado,Código de país
8,remote_ratio,int64,Porcentaje de trabajo remoto,"%: 0, 50 o 100"
9,company_location,object,País de ubicación de la empresa,Código de país


In [10]:
# Comprobar los archivos exportados y los totales de las tablas.
archivos_esperados = [
    ruta_preparado,
    ruta_resumen,
    CARPETA_RESULTADOS / "pregunta_principal_salarios_por_grupo.csv",
    CARPETA_RESULTADOS / "pregunta_apoyo_1_registros_por_anio_experiencia_empleo.csv",
    CARPETA_RESULTADOS / "pregunta_apoyo_2_cargos_por_registros_y_mediana.csv",
    CARPETA_RESULTADOS / "pregunta_apoyo_3_salario_mediano_por_anio.csv",
]

for ruta in archivos_esperados:
    if not ruta.is_file():
        raise FileNotFoundError(f"No se generó el archivo: {ruta}")
    print(f"OK: {ruta.relative_to(ROOT)}")

# Comprobar que cada tabla CSV conserva el conteo del notebook.
for nombre, tabla in tablas_resultados.items():
    ruta = CARPETA_RESULTADOS / f"{nombre}.csv"
    tabla_csv = pd.read_csv(ruta)

    assert len(tabla_csv) == len(tabla), (
        f"El número de filas no coincide en {ruta.name}."
    )
    assert int(tabla_csv["cantidad_registros"].sum()) == len(df), (
        f"Los conteos de {ruta.name} no suman {len(df)}."
    )

# Comprobar también las dimensiones del dataset exportado.
df_csv = pd.read_csv(ruta_preparado)
assert df_csv.shape == df.shape, (
    f"Dimensiones distintas: CSV {df_csv.shape}, DataFrame {df.shape}."
)

print("\nVerificación correcta: están los seis archivos y los conteos suman", len(df))

OK: src/Entrega_3/datos/demo-salaries-preparado.csv
OK: src/Entrega_3/resultados/resumen_dataframe.txt
OK: src/Entrega_3/resultados/pregunta_principal_salarios_por_grupo.csv
OK: src/Entrega_3/resultados/pregunta_apoyo_1_registros_por_anio_experiencia_empleo.csv
OK: src/Entrega_3/resultados/pregunta_apoyo_2_cargos_por_registros_y_mediana.csv
OK: src/Entrega_3/resultados/pregunta_apoyo_3_salario_mediano_por_anio.csv

Verificación correcta: están los seis archivos y los conteos suman 3755
